In [0]:
%sql
SELECT
  late_delivery AS pedido_atrasado,
  COUNT(*) AS qtd_pedidos,
  ROUND(avg(warehouse_processing_hours), 2) AS mediana_processamento_hrs
FROM workspace.default.entregas_tratadas
GROUP BY late_delivery
ORDER BY late_delivery;

Os pedidos atrasados apresentaram tempo médio de processamento de 16,59 horas, contra 11,60 horas nos pedidos entregues no prazo. Essa diferença indica uma associação entre o tempo de processamento e a ocorrência de atrasos, sugerindo que a etapa de preparação dos pedidos merece atenção operacional.


In [0]:
%sql
SELECT
    delivery_attempts AS tentativas_entrega,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY delivery_attempts
ORDER BY delivery_attempts;

O percentual de atraso aumentou de 55,98% nos pedidos com uma tentativa para 57,44% nos pedidos com duas tentativas, chegando a 75% nos pedidos com três tentativas. O resultado indica uma associação entre o número de tentativas e a ocorrência de atrasos. Como apenas 292 pedidos exigiram três tentativas, esse resultado deve ser interpretado com cautela, sem afirmar uma relação de causa e efeito.

In [0]:
%sql
-- 1. distancia percorrida e atrasos

SELECT
    CASE
        WHEN distance_km <500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END AS faixa_distancia,
    COUNT(*) AS total_pedidos,
    SUM(CASE            
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2   
    ) AS percentual_atrasos
FROM workspace.default.entregas_tratadas
GROUP BY 
    CASE
        WHEN distance_km <500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END
ORDER BY MIN(distance_km)

O percentual de atraso aumentou conforme a faixa de distância percorrida, passando de 36,55% nos pedidos de até 499 km para 63,89% nos pedidos de 1.500 km ou mais. Os resultados indicam uma associação entre maiores distâncias e maior frequência de atrasos. Recomenda-se investigar se as rotas mais longas exigem ajustes nos prazos prometidos ou no planejamento logístico.


In [0]:
%sql
-- 2. condicoes climaticas e atrasos

SELECT
    weather_condition AS condicao_climatica,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY weather_condition
ORDER BY percentual_atraso DESC;

Os pedidos registrados sob condições de neve e tempestade apresentaram os maiores percentuais de atraso, com 93,51% e 91%, respectivamente. Em contrapartida, pedidos com céu limpo apresentaram 34,97% de atraso. Os resultados indicam uma associação entre condições climáticas adversas e maior frequência de atrasos, sugerindo a necessidade de avaliar ajustes nos prazos e no planejamento logístico em períodos de condições climáticas desfavoráveis.
